## CRITICAL UPDATE: Proper Data Partitioning Implementation

### What Was Fixed:

#### Before (INCORRECT):
- **CapsNet**: Used standard `KFold` with uniform random splits
- **LSTM**: Used `TimeSeriesSplit` with default uniform splits
- **Result**: Inconsistent with thesis methodology

#### After (CORRECT):
- **Both CapsNet & LSTM**: Use custom expanding window splits
- **Chunk distribution**: [15%, 15%, 15%, 15%, 20%, 20%]
- **Time-ordered**: No shuffling, maintains temporal sequence
- **Expanding window**: Training data grows with each fold
- **Result**: Fully compliant with thesis requirements

---

### Fold Distribution (From Thesis):

```
Total Data: 100% (80% learning set for CV + 20% holdout test set)
CV uses the 80% learning set only

Chunks: [15%, 15%, 15%, 15%, 20%, 20%] = 100% of learning set

Fold 1: Train on Chunk 1 (15%)     → Validate on Chunk 2 (15%)
Fold 2: Train on Chunks 1-2 (30%)  → Validate on Chunk 3 (15%)
Fold 3: Train on Chunks 1-3 (45%)  → Validate on Chunk 4 (15%)
Fold 4: Train on Chunks 1-4 (60%)  → Validate on Chunk 5 (20%)
Fold 5: Train on Chunks 1-5 (80%)  → Validate on Chunk 6 (20%)
```

---

### Implementation Details:

#### New Method: `create_expanding_window_splits()`
- Calculates exact chunk boundaries based on data length
- Returns list of (train_idx, val_idx) tuples for each fold
- Applied to **both CapsNet and LSTM** for consistency

#### Updated Methods:
1. `train_capsnet_fold()` - Now uses expanding window splits
2. `train_lstm_fold()` - Now uses expanding window splits
3. `extract_capsnet_features_fold()` - Uses same splits for extraction

---

### Why This Matters:

1. **Thesis Compliance**: Exactly matches the described methodology
2. **Time Series Integrity**: Preserves temporal order (no shuffling)
3. **Expanding Window**: Mimics real-world scenario where more data accumulates
4. **Consistency**: Both models see the same data splits
5. **Evaluation**: Each fold's validation set is independent

---

### Important Notes:

- The splits are **deterministic** (no random shuffling)
- Data must be **pre-sorted by timestamp** before splitting
- The 20% holdout test set is **never used in CV** (kept separate)
- Each fold's validation chunk is **only used once**

---

# Complete CapsNet + LSTM + LightGBM Pipeline with 5-Fold Cross-Validation

This notebook implements a complete machine learning pipeline with **PROPER PER-FOLD EXECUTION**:

## Correct Workflow (Per Fold):
For **EACH fold (1 to 5)**:
1. Train CapsNet on fold training data
2. Train LSTM on fold training data
3. Extract CapsNet features from fold validation data
4. Extract LSTM features from fold validation data
5. Fuse features (CapsNet + LSTM)
6. Train LightGBM on fused features
7. Evaluate on fold validation data

## Key Features:
- Proper cross-validation: Complete workflow per fold
- No data leakage between folds
- Memory efficient: One fold at a time
- Comprehensive metrics and visualizations

**Author:** Thesis Research  
**Date:** October 2025  
**Purpose:** Air Quality Prediction using Multi-Modal Deep Learning

## 1. Environment Setup and Imports

Import all required libraries and set up the environment for the complete ML pipeline.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import traceback
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Tuple, Optional
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import KFold, train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import lightgbm as lgb
import gc
warnings.filterwarnings('ignore')

# Set up matplotlib for inline plotting
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("All libraries imported successfully!")

# Add src directory to Python path
current_dir = Path.cwd()
src_dir = current_dir / "src"
sys.path.insert(0, str(src_dir))
sys.path.insert(0, str(current_dir))

print(f"Current directory: {current_dir}")
print(f"Source directory: {src_dir}")

# Verify CUDA availability and setup memory optimization
try:
    import torch
    
    # CRITICAL: Memory optimization for 4GB GPU
    # Enable memory-efficient allocator
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    
    # Set to use GPU 0 (your only GPU)
    os.environ['CUDA_VISIBLE_DEVICES'] = '0'
    
    # Enable deterministic mode to reduce memory overhead
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    
    # Enable memory efficient operations
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    
    if torch.cuda.is_available():
        # Clear any existing GPU cache
        torch.cuda.empty_cache()
        gc.collect()
        
        print(f"CUDA available! Using GPU 0")
        print(f"Device: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        print(f"Memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.3f} GB")
        print(f"Memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.3f} GB")
        print(f"Memory optimization enabled for 4GB GPU")
    else:
        print("WARNING: CUDA not available, using CPU")
except ImportError:
    print("WARNING: PyTorch not found, ensure it's installed for GPU acceleration")

In [ ]:
# Import custom model classes
import importlib
try:
    # Force reload to get latest code changes
    if 'src.training.capsnet_trainer' in sys.modules:
        importlib.reload(sys.modules['src.training.capsnet_trainer'])
    if 'src.lstm.lstm_temporal_feature_generator' in sys.modules:
        importlib.reload(sys.modules['src.lstm.lstm_temporal_feature_generator'])
    
    from src.training.capsnet_trainer import CapsNetTrainer, AirQualityDataset
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator
    print("Custom model classes imported successfully!")
    print("Modules reloaded with latest code changes!")
except ImportError as e:
    print(f"ERROR: Import error: {e}")
    print("WARNING: Please ensure all model files are in the correct locations")
    print("Trying alternative import paths...")
    try:
        # Try without the src prefix (if src is in sys.path)
        import sys
        from training.capsnet_trainer import CapsNetTrainer, AirQualityDataset
        from lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator
        print("Custom model classes imported successfully (alternative path)!")
    except ImportError as e2:
        print(f"ERROR: Alternative import also failed: {e2}")
        print("WARNING: Please check that:")
        print("   1. src/training/capsnet_trainer.py exists")
        print("   2. src/lstm/lstm_temporal_feature_generator.py exists")
        print("   3. All __init__.py files are present in the directories")

## 2. Pipeline Configuration

Define the CompleteMLPipeline class with all necessary methods for the end-to-end machine learning pipeline.

In [ ]:
class CompleteMLPipeline:
    """Complete ML Pipeline with CapsNet + LSTM + LightGBM"""
    
    def __init__(self, day_folder: str, output_dir: str = "per_day_outputs", fast_mode: bool = False):
        self.day_folder = day_folder
        self.output_dir = output_dir
        self.n_folds = 2 if fast_mode else 5  # 2 folds for ultra-fast testing, 5 for full CV
        self.fast_mode = fast_mode
        self.device = 'cuda:0'  # Using GPU 0 (your only GPU)
        
        # Clear GPU memory before initialization
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
                print(f"GPU memory cleared before pipeline initialization")
        except:
            pass
        
        # Create organized output directories
        self.setup_directories()
        
        # Results storage
        self.fold_results = []
        self.capsnet_features = {}
        self.lstm_features = {}
        self.final_results = {}
        
        print(f"\nComplete ML Pipeline initialized for {day_folder}")
        print(f"Output directory: {output_dir}")
        print(f"Using {self.n_folds}-fold cross-validation")
        print(f"Using device: {self.device} (GPU 0 - RTX 3050 4GB)")
        if fast_mode:
            print(f"⚡ ULTRA-FAST MODE: 2-fold CV with 1 epoch for rapid testing")
        else:
            print(f"🎯 FULL MODE: 5-fold cross-validation for robust evaluation")
    
    def setup_directories(self):
        """Create organized directory structure"""
        directories = [
            self.output_dir,
            f"{self.output_dir}/models/capsnet",
            f"{self.output_dir}/models/lstm", 
            f"{self.output_dir}/models/lightgbm",
            f"{self.output_dir}/features/capsnet",
            f"{self.output_dir}/features/lstm",
            f"{self.output_dir}/features/fused",
            f"{self.output_dir}/results",
            f"{self.output_dir}/plots",
            f"{self.output_dir}/cv_folds"
        ]
        
        for directory in directories:
            os.makedirs(directory, exist_ok=True)
        
        print(f"Directory structure created!")

print("CompleteMLPipeline class defined!")

In [ ]:
def create_expanding_window_splits(self, data_length: int):
    """
    Create 5-fold expanding window splits with custom chunk sizes.
    
    Chunk distribution: [15%, 15%, 15%, 15%, 20%, 20%]
    Total: 6 chunks for 5 folds
    
    Fold 1: Train on 15% → Validate on 15%
    Fold 2: Train on 30% → Validate on 15%
    Fold 3: Train on 45% → Validate on 15%
    Fold 4: Train on 60% → Validate on 20%
    Fold 5: Train on 80% → Validate on 20%
    """
    # Define chunk sizes (percentages)
    chunk_percentages = [0.15, 0.15, 0.15, 0.15, 0.20, 0.20]
    
    # Calculate chunk indices
    chunk_indices = [0]
    cumulative = 0
    for pct in chunk_percentages:
        cumulative += pct
        chunk_indices.append(int(data_length * cumulative))
    
    # Create fold splits (expanding window)
    splits = []
    for fold in range(5):  # 5 folds
        train_end = chunk_indices[fold + 1]
        val_start = chunk_indices[fold + 1]
        val_end = chunk_indices[fold + 2]
        
        train_idx = list(range(0, train_end))
        val_idx = list(range(val_start, val_end))
        
        splits.append((train_idx, val_idx))
    
    return splits

# Add the method to the class
CompleteMLPipeline.create_expanding_window_splits = create_expanding_window_splits
print("Custom expanding window split method added!")

## Data Partitioning: 5-Fold Expanding Window with Custom Chunk Sizes

### Chunk Distribution: [15%, 15%, 15%, 15%, 20%, 20%]

According to the thesis methodology:

| Fold | Training Data | Validation Data | Train Size | Val Size |
|------|---------------|-----------------|------------|----------|
| 1    | Chunk 1       | Chunk 2         | 15%        | 15%      |
| 2    | Chunks 1-2    | Chunk 3         | 30%        | 15%      |
| 3    | Chunks 1-3    | Chunk 4         | 45%        | 15%      |
| 4    | Chunks 1-4    | Chunk 5         | 60%        | 20%      |
| 5    | Chunks 1-5    | Chunk 6         | 80%        | 20%      |

### Key Points:
- **Expanding Window**: Training data grows with each fold
- **Time-Ordered**: Data maintains temporal sequence (no shuffling)
- **Custom Chunks**: First 4 chunks are 15%, last 2 chunks are 20%
- **Applied to Both**: Same split used for CapsNet AND LSTM
- **80/20 Split**: Total learning set is 80%, holdout test set is 20%

### Within Each Fold:
1. **Feature Extraction**: CapsNet (spatial) + LSTM (temporal)
2. **Feature Fusion**: Concatenate spatial + temporal features
3. **Final Regression**: LightGBM trained on fused features
4. **Evaluation**: Metrics calculated on validation chunk

## 3. Data Loading and Hyperparameter Setup

Load best hyperparameters from previous tuning results and set up default parameters.

In [ ]:
# Add hyperparameter loading method to the pipeline class
def load_best_hyperparameters(self, model_type: str) -> Dict:
    """Load best hyperparameters from previous tuning results"""
    print(f"Loading best hyperparameters for {model_type}...")
    
    # Look for hyperparameter files
    import glob
    
    # Extract the date part from day_folder (e.g., "7_24_data" -> "7_24")
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    
    # Define different patterns for different model types
    if model_type == "capsnet":
        param_patterns = [
            f"outputs/capsnet/hyperparameters/basic/best_params_basic_{self.day_folder}_*.json",
            f"outputs/capsnet/hyperparameters/advanced/best_params_advanced_{self.day_folder}_*.json",
            f"best_params_capsnet_{self.day_folder}.json",
            f"best_params_capsnet.json"
        ]
    elif model_type == "lstm":
        param_patterns = [
            f"src/lstm/{date_part}_best_params.json",  # Matches: 7_24_best_params.json
            f"src/lstm/{self.day_folder}_best_params.json",  # Alternative: 7_24_data_best_params.json
            f"src/lstm/best_params_{date_part}.json",  # Another format: best_params_7_24.json
            f"outputs/lstm/hyperparameters/best_params_{self.day_folder}_*.json",  # Fallback
            f"best_params_lstm_{self.day_folder}.json",
            f"best_params_lstm.json"
        ]
    else:
        param_patterns = [
            f"best_params_{model_type}_{self.day_folder}.json",
            f"best_params_{model_type}.json"
        ]
    
    best_params = None
    for pattern in param_patterns:
        files = glob.glob(pattern)
        if files:
            # Use the most recent file
            latest_file = max(files, key=os.path.getmtime)
            try:
                with open(latest_file, 'r') as f:
                    best_params = json.load(f)
                print(f"Loaded parameters from: {latest_file}")
                break
            except Exception as e:
                print(f"Error loading {latest_file}: {e}")
                continue
    
    if best_params is None:
        print(f"No saved hyperparameters found for {model_type}, using defaults")
        # Default parameters
        if model_type == "capsnet":
            best_params = {
                'learning_rate': 0.001,
                'dropout_rate': 0.3,
                'feature_dim': 128,
                'optimizer_type': 'adam',
                'weight_decay': 0.0001,
                'batch_size': 8
            }
        elif model_type == "lstm":
            best_params = {
                'learning_rate': 0.001,
                'hidden_size': 128,
                'num_layers': 2,
                'dropout': 0.2,
                'batch_size': 32
            }
    
    print(f"{model_type.upper()} parameters: {best_params}")
    return best_params

# Add the method to the class
CompleteMLPipeline.load_best_hyperparameters = load_best_hyperparameters
print("Hyperparameter loading method added to pipeline class!")

## 4. CapsNet Cross-Validation Training

Implement 5-fold cross-validation training for the CapsNet model.

In [ ]:
def train_capsnet_cv(self, capsnet_params: Dict) -> Dict[int, str]:
    """Train CapsNet with 5-fold cross-validation"""
    print(f"\nTraining CapsNet with {self.n_folds}-fold CV...")
    print(f"   Parameters: {capsnet_params}")
    print(f"   Using EfficientCapsNet with Self-Attention Routing")
    
    # Clear GPU memory before training
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
            print(f"GPU memory cleared before training")
            print(f"Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    except:
        pass
    
    # Initialize trainer with GPU 0
    trainer = CapsNetTrainer(
        input_size=256,
        feature_dim=capsnet_params.get('feature_dim', 128),
        device=self.device  # Use GPU 0
    )
    
    # Load data
    learning_df, patch_metadata_df = trainer.load_day_data(self.day_folder)
    
    # Setup K-fold CV
    kfold = KFold(n_splits=self.n_folds, shuffle=True, random_state=42)
    fold_models = {}
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(learning_df)):
        print(f"\nCapsNet Fold {fold + 1}/{self.n_folds}")
        print("-" * 40)
        
        # Clear GPU memory before each fold
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                gc.collect()
                mem_allocated = torch.cuda.memory_allocated(0) / 1e9
                mem_reserved = torch.cuda.memory_reserved(0) / 1e9
                print(f"GPU memory cleared for fold {fold+1}")
                print(f"Allocated: {mem_allocated:.3f} GB | Reserved: {mem_reserved:.3f} GB")
        except:
            pass
        
        # Split data for this fold
        train_df = learning_df.iloc[train_idx].reset_index(drop=True)
        val_df = learning_df.iloc[val_idx].reset_index(drop=True)
        
        # Create datasets
        train_dataset = AirQualityDataset(train_df, patch_metadata_df, self.day_folder, 'train')
        val_dataset = AirQualityDataset(val_df, patch_metadata_df, self.day_folder, 'val')
        
        print(f"   Training samples: {len(train_dataset)}")
        print(f"   Validation samples: {len(val_dataset)}")
        
        # Create model for this fold
        # Filter out parameters that are already passed to __init__ or create_model directly
        model_params = {k: v for k, v in capsnet_params.items() 
                       if k not in ['batch_size', 'feature_dim', 'learning_rate', 'weight_decay', 'optimizer_type']}
        
        print(f"Creating EfficientCapsNet model for fold {fold+1}...")
        # Use efficient=True for EfficientCapsNet with Self-Attention Routing
        trainer.create_model(use_efficient=True, **model_params)
        
        trainer.setup_training(
            learning_rate=capsnet_params.get('learning_rate', 0.001),
            weight_decay=capsnet_params.get('weight_decay', 1e-4),
            optimizer_type=capsnet_params.get('optimizer_type', 'adam')
        )
        
        # Train (adaptive epochs based on mode)
        epochs = 3 if self.fast_mode else 10
        best_loss = trainer.train(
            train_dataset, val_dataset,
            epochs=epochs,
            batch_size=capsnet_params.get('batch_size', 8),  # Use batch size from params
            day_folder=f"{self.day_folder}_fold_{fold+1}"
        )
        
        # Save fold model
        fold_model_path = f"{self.output_dir}/models/capsnet/capsnet_efficient_fold_{fold+1}_{self.day_folder}.pth"
        trainer.save_model(fold_model_path, 30, best_loss)
        fold_models[fold+1] = fold_model_path
        
        print(f"   Fold {fold+1} completed! Best loss: {best_loss:.4f}")
        
        # Clear memory after fold
        try:
            import torch
            if torch.cuda.is_available():
                del trainer.model
                torch.cuda.empty_cache()
                gc.collect()
                print(f"Memory cleared after fold {fold+1}")
        except:
            pass
    
    print(f"\nCapsNet {self.n_folds}-fold CV completed!")
    return fold_models

# Add the method to the class
CompleteMLPipeline.train_capsnet_cv = train_capsnet_cv
print("CapsNet cross-validation training method added (using EfficientCapsNet)!")

## 5. LSTM Cross-Validation Training

Implement 5-fold cross-validation training for the LSTM model.

In [ ]:
def train_lstm_cv(self, lstm_params: Dict) -> Dict[int, str]:
    """Train LSTM with 5-fold expanding window cross-validation and extract features"""
    print(f"\nTraining LSTM with {self.n_folds}-fold expanding window CV...")
    print(f"   Parameters: {lstm_params}")

    # Load temporal data and targets
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator, TemporalDataLoader
    data_loader = TemporalDataLoader()
    
    # Extract date part from day_folder (e.g., "7_24_data" -> "7_24")
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    temporal_data, targets, feature_names = data_loader.load_temporal_data(date_part)

    n_total = len(temporal_data)
    n_learning = int(n_total * 0.8)
    learning_temporal = temporal_data[:n_learning]
    learning_targets = targets[:n_learning]

    from sklearn.model_selection import TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=self.n_folds)
    lstm_generator = LSTMTemporalFeatureGenerator(lstm_params)

    fold_models = {}
    for fold, (train_idx, val_idx) in enumerate(tscv.split(learning_temporal)):
        print(f"\nLSTM Fold {fold + 1}/{self.n_folds}")
        print("-" * 40)

        train_temporal = learning_temporal[train_idx]
        train_targets = learning_targets[train_idx]
        val_temporal = learning_temporal[val_idx]
        val_targets = learning_targets[val_idx]

        # Train LSTM and extract features
        train_temp_features, val_temp_features, _, _ = lstm_generator.train_and_extract_features(
            train_temporal, train_targets, val_temporal, val_targets
        )

        # Save features for this fold
        features_path = f"{self.output_dir}/features/lstm_fold_{fold+1}_{self.day_folder}.npz"
        np.savez(features_path,
                 train_features=train_temp_features,
                 val_features=val_temp_features,
                 train_targets=train_targets,
                 val_targets=val_targets)
        fold_models[fold+1] = features_path

        print(f"   LSTM Fold {fold+1} completed! Features saved to {features_path}")

    print(f"\nLSTM {self.n_folds}-fold CV completed!")
    return fold_models

# Add the method to the class
CompleteMLPipeline.train_lstm_cv = train_lstm_cv
print("LSTM cross-validation training method updated!")

In [ ]:
def train_capsnet_fold(self, capsnet_params: Dict, fold: int) -> str:
    """Train CapsNet for a specific fold using expanding window splits"""
    print(f"      Initializing CapsNet training...")
    
    # Clear GPU memory
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()
    except:
        pass
    
    # Initialize trainer
    trainer = CapsNetTrainer(
        input_size=256,
        feature_dim=capsnet_params.get('feature_dim', 128),
        device=self.device
    )
    
    # Load data
    learning_df, patch_metadata_df = trainer.load_day_data(self.day_folder)
    
    # Use custom expanding window splits (15%, 15%, 15%, 15%, 20%, 20%)
    splits = self.create_expanding_window_splits(len(learning_df))
    train_idx, val_idx = splits[fold-1]
    
    # Split data for this fold
    train_df = learning_df.iloc[train_idx].reset_index(drop=True)
    val_df = learning_df.iloc[val_idx].reset_index(drop=True)
    
    # Create datasets
    train_dataset = AirQualityDataset(train_df, patch_metadata_df, self.day_folder, 'train')
    val_dataset = AirQualityDataset(val_df, patch_metadata_df, self.day_folder, 'val')
    
    print(f"         Training samples: {len(train_dataset)}")
    print(f"         Validation samples: {len(val_dataset)}")
    
    # Create model
    model_params = {k: v for k, v in capsnet_params.items() 
                   if k not in ['batch_size', 'feature_dim', 'learning_rate', 'weight_decay', 'optimizer_type']}
    
    trainer.create_model(use_efficient=True, **model_params)
    trainer.setup_training(
        learning_rate=capsnet_params.get('learning_rate', 0.001),
        weight_decay=capsnet_params.get('weight_decay', 1e-4),
        optimizer_type=capsnet_params.get('optimizer_type', 'adam')
    )
    
    # Train
    epochs = 1 if self.fast_mode else 10
    best_loss = trainer.train(
        train_dataset, val_dataset,
        epochs=epochs,
        batch_size=capsnet_params.get('batch_size', 8),
        day_folder=f"{self.day_folder}_fold_{fold}"
    )
    
    # Save model
    fold_model_path = f"{self.output_dir}/models/capsnet/capsnet_fold_{fold}_{self.day_folder}.pth"
    trainer.save_model(fold_model_path, 30, best_loss)
    
    print(f"         CapsNet trained! Loss: {best_loss:.4f} (Epochs: {epochs})")
    
    # Clear memory
    try:
        import torch
        if torch.cuda.is_available():
            del trainer.model
            torch.cuda.empty_cache()
            gc.collect()
    except:
        pass
    
    return fold_model_path

def train_lstm_fold(self, lstm_params: Dict, fold: int) -> str:
    """Train LSTM for a specific fold using expanding window splits (training only)"""
    print(f"      Initializing LSTM training...")
    
    # Load temporal data
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator, TemporalDataLoader
    data_loader = TemporalDataLoader()
    
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    temporal_data, targets, feature_names = data_loader.load_temporal_data(date_part)
    
    n_total = len(temporal_data)
    n_learning = int(n_total * 0.8)
    learning_temporal = temporal_data[:n_learning]
    learning_targets = targets[:n_learning]
    
    # Use custom expanding window splits (15%, 15%, 15%, 15%, 20%, 20%)
    splits = self.create_expanding_window_splits(len(learning_temporal))
    train_idx, val_idx = splits[fold-1]
    
    train_temporal = learning_temporal[train_idx]
    train_targets = learning_targets[train_idx]
    val_temporal = learning_temporal[val_idx]
    val_targets = learning_targets[val_idx]
    
    print(f"         Training samples: {len(train_temporal)}")
    print(f"         Validation samples: {len(val_temporal)}")
    
    # Train LSTM only (no feature extraction yet)
    # Use fewer epochs in fast mode for quicker testing
    epochs = 1 if self.fast_mode else 10
    lstm_generator = LSTMTemporalFeatureGenerator(lstm_params)
    lstm_generator.train_model(train_temporal, train_targets, val_temporal, val_targets, epochs=epochs)
    
    # Save the trained model
    model_path = f"{self.output_dir}/models/lstm/lstm_fold_{fold}_{self.day_folder}.pth"
    lstm_generator.save_model(model_path)
    
    print(f"         LSTM trained and saved! (Epochs: {epochs})")
    
    return model_path

def extract_lstm_features_fold(self, model_path: str, fold: int, lstm_params: Dict) -> pd.DataFrame:
    """Extract LSTM features for a specific fold using trained model"""
    print(f"      Extracting LSTM features...")
    
    # Load temporal data
    from src.lstm.lstm_temporal_feature_generator import LSTMTemporalFeatureGenerator, TemporalDataLoader
    data_loader = TemporalDataLoader()
    
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    temporal_data, targets, feature_names = data_loader.load_temporal_data(date_part)
    
    n_total = len(temporal_data)
    n_learning = int(n_total * 0.8)
    learning_temporal = temporal_data[:n_learning]
    learning_targets = targets[:n_learning]
    
    # Use custom expanding window splits (same as training)
    splits = self.create_expanding_window_splits(len(learning_temporal))
    train_idx, val_idx = splits[fold-1]
    
    val_temporal = learning_temporal[val_idx]
    val_targets = learning_targets[val_idx]
    
    # Load trained model and extract features
    lstm_generator = LSTMTemporalFeatureGenerator(lstm_params)
    lstm_generator.load_model(model_path)
    
    # Extract features from validation data
    val_features = lstm_generator.extract_features(val_temporal)
    
    # Adjust targets to match features length (account for timesteps)
    timesteps = lstm_params.get('timesteps', 60)
    adjusted_targets = val_targets[timesteps:]
    
    # Create DataFrame
    feature_cols = [f'lstm_f_{i}' for i in range(val_features.shape[1])]
    lstm_df = pd.DataFrame(val_features, columns=feature_cols)
    lstm_df['pm2.5'] = adjusted_targets[:len(val_features)]
    
    # Save features
    features_path = f"{self.output_dir}/features/lstm/lstm_features_fold_{fold}_{self.day_folder}.csv"
    lstm_df.to_csv(features_path, index=False)
    
    print(f"         LSTM features extracted! Shape: {lstm_df.shape}")
    
    return lstm_df

# Add the methods to the class
CompleteMLPipeline.train_capsnet_fold = train_capsnet_fold
CompleteMLPipeline.train_lstm_fold = train_lstm_fold
CompleteMLPipeline.extract_lstm_features_fold = extract_lstm_features_fold
print("Individual fold training methods added!")

## 6. Feature Extraction from All Folds

Extract features from trained CapsNet and LSTM models for each cross-validation fold.

In [ ]:
def extract_features_cv(self):
    """Extract features using cross-validation for both training and test"""
    print("\n" + "="*70)
    print("PHASE 2: FEATURE EXTRACTION")
    print("="*70)
    
    # Load all trained models
    capsnet_models = {}
    lstm_models = {}
    
    for fold in range(1, self.n_folds + 1):
        capsnet_path = f"{self.output_dir}/models/capsnet/capsnet_fold_{fold}_{self.day_folder}.pth"
        lstm_path = f"{self.output_dir}/models/lstm/lstm_fold_{fold}_{self.day_folder}.pth"
        
        if not os.path.exists(capsnet_path) or not os.path.exists(lstm_path):
            raise FileNotFoundError(f"Models not found for fold {fold}")
            
        capsnet_models[fold] = capsnet_path
        lstm_models[fold] = lstm_path
    
    # Extract features for each fold
    for fold in range(1, self.n_folds + 1):
        print(f"\n{'='*70}")
        print(f"Extracting Features - Fold {fold}/{self.n_folds}")
        print(f"{'='*70}")
        
        # Extract features using trained models
        capsnet_features = self.extract_capsnet_features_fold(capsnet_models[fold], fold)
        lstm_features = self.extract_lstm_features_fold(lstm_models[fold], fold)
        
        # Fuse features
        fused_features = self.fuse_features_fold(capsnet_features, lstm_features, fold)
        
        print(f"   Fold {fold} feature extraction complete!")

def extract_capsnet_features_fold(self, model_path: str, fold: int, capsnet_params: Dict = None) -> pd.DataFrame:
    """Extract CapsNet features for a specific fold using expanding window splits
    
    Args:
        model_path: Path to the trained CapsNet model
        fold: Fold number (1-indexed)
        capsnet_params: Optional CapsNet parameters (for compatibility, not used in extraction)
    """
    from components.capsnet.capsnet_trainer import CapsNetTrainer, AirQualityDataset, custom_collate_fn
    
    print(f"      Loading CapsNet model from: {model_path}")
    
    # Load checkpoint first to get the correct feature_dim
    checkpoint = torch.load(model_path, map_location=self.device)
    feature_dim = checkpoint.get('feature_dim', 128)
    input_size = checkpoint.get('input_size', 256)
    
    print(f"         Model config: input_size={input_size}, feature_dim={feature_dim}")
    
    # Initialize trainer with the same configuration as training
    trainer = CapsNetTrainer(
        input_size=input_size,
        feature_dim=feature_dim,
        device=self.device
    )
    
    # Load data using the same method as training
    learning_df, patch_metadata_df = trainer.load_day_data(self.day_folder)
    
    # Use custom expanding window splits (same as training)
    splits = self.create_expanding_window_splits(len(learning_df))
    train_idx, val_idx = splits[fold-1]
    
    # Split data for this fold
    val_df = learning_df.iloc[val_idx].reset_index(drop=True)
    
    # Create validation dataset
    val_dataset = AirQualityDataset(val_df, patch_metadata_df, self.day_folder, 'val')
    
    print(f"         Validation samples: {len(val_dataset)}")
    
    # Create model with correct architecture - CRITICAL: use_efficient=True!
    trainer.create_model(use_efficient=True)  # ✅ Must match training architecture
    
    # Load model weights from checkpoint
    trainer.feature_extractor.load_state_dict(checkpoint['model_state_dict'])
    trainer.feature_extractor.to(self.device)
    trainer.feature_extractor.eval()
    
    print(f"         Model loaded successfully (epoch {checkpoint.get('epoch', 'N/A')})")
    
    # Extract features with larger batch size for speed
    batch_size = 16 if self.fast_mode else 8
    
    # Get all data from dataset - IMPORTANT: Use custom_collate_fn
    from torch.utils.data import DataLoader
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False,
        collate_fn=custom_collate_fn  # ✅ Required for proper metadata handling
    )
    
    all_features = []
    all_metadata = []
    
    with torch.no_grad():
        # AirQualityDataset returns 3 values: image, pm25, metadata
        for batch_images, batch_pm25, batch_metadata in val_loader:
            batch_images = batch_images.to(self.device)
            # Extract features from the model
            features = trainer.feature_extractor(batch_images)
            all_features.append(features.cpu().numpy())
            
            # Store metadata (batch_metadata is already a list of dicts)
            all_metadata.extend(batch_metadata)
    
    # Concatenate all features
    features = np.concatenate(all_features, axis=0)
    
    # Convert to DataFrame
    feature_df = pd.DataFrame(features, columns=[f'caps_f_{i}' for i in range(features.shape[1])])
    
    # Add metadata if available
    if all_metadata and len(all_metadata) > 0:
        # Check if metadata is dict (not string)
        if isinstance(all_metadata[0], dict):
            feature_df['pm25'] = [m.get('pm2.5', 0) for m in all_metadata]
            # Add other metadata fields if needed
            if 'lat' in all_metadata[0]:
                feature_df['lat'] = [m.get('lat', 0) for m in all_metadata]
            if 'lon' in all_metadata[0]:
                feature_df['lon'] = [m.get('lon', 0) for m in all_metadata]
    
    # Save features
    feature_path = f"{self.output_dir}/features/capsnet/capsnet_features_fold_{fold}_{self.day_folder}.csv"
    feature_df.to_csv(feature_path, index=False)
    
    print(f"         Extracted {len(feature_df)} samples with {features.shape[1]} features (batch_size={batch_size})")
    
    return feature_df

def extract_lstm_features_fold(self, model_path: str, fold: int, lstm_params: Dict = None) -> pd.DataFrame:
    """Extract LSTM features for a specific fold using expanding window splits
    
    Args:
        model_path: Path to the trained LSTM model
        fold: Fold number (1-indexed)
        lstm_params: Optional LSTM parameters (for compatibility, not used in extraction)
    """
    from components.lstm.lstm_temporal_gen import LSTMTemporalFeatureGenerator, TemporalDataLoader
    
    print(f"      Loading LSTM model from: {model_path}")
    
    # Load temporal data
    data_loader = TemporalDataLoader()
    date_part = self.day_folder.replace('_data', '') if '_data' in self.day_folder else self.day_folder
    temporal_data, targets, feature_names = data_loader.load_temporal_data(date_part)
    
    # Get learning set (80%)
    n_total = len(temporal_data)
    n_learning = int(n_total * 0.8)
    learning_temporal = temporal_data[:n_learning]
    
    # Use custom expanding window splits (same as training)
    splits = self.create_expanding_window_splits(len(learning_temporal))
    train_idx, val_idx = splits[fold-1]
    
    # Use validation set for feature extraction
    X_val = learning_temporal[val_idx]
    
    # Load LSTM model - note: no device parameter!
    lstm_gen = LSTMTemporalFeatureGenerator()
    lstm_gen.load_model(model_path)
    
    # Extract features - NOTE: extract_features() doesn't accept batch_size parameter
    # The batch processing is handled internally by the method
    lstm_features = lstm_gen.extract_features(X_val)
    
    # Convert to DataFrame
    feature_df = pd.DataFrame(lstm_features, columns=[f'lstm_f_{i}' for i in range(lstm_features.shape[1])])
    
    # Save features
    feature_path = f"{self.output_dir}/features/lstm/lstm_features_fold_{fold}_{self.day_folder}.csv"
    feature_df.to_csv(feature_path, index=False)
    
    print(f"         Extracted {len(feature_df)} samples")
    
    return feature_df

# Add methods to the class
CompleteMLPipeline.extract_features_cv = extract_features_cv
CompleteMLPipeline.extract_capsnet_features_fold = extract_capsnet_features_fold
CompleteMLPipeline.extract_lstm_features_fold = extract_lstm_features_fold
print("Feature extraction methods added (with EfficientCapsNet support)!")

## 7. Feature Fusion and LightGBM Training

Fuse CapsNet and LSTM features for each fold and train LightGBM models.

### ⚠️ CRITICAL UPDATE: Patch Aggregation for Feature Fusion

**Problem Identified:**
- CapsNet extracts **patch-level features**: 84,770 samples (10 patches × ~8,477 locations)
- LSTM extracts **location-level features**: 7,694 samples (1 per location-timestamp)
- Previous implementation: Truncated first 7,694 CapsNet patches (randomly discarding 91% of data!)

**Solution Implemented:**
- **Aggregate CapsNet patches by location** using mean pooling
- Groups patches by `(lat, lon)` coordinates
- Reduces 84,770 patches → ~7,694 locations
- Now both branches have **same first dimension** as required by thesis

**Why This Matters for Your Thesis:**
Your thesis states: *"Both branches' outputs are NumPy arrays with the same first dimension"*
- ✅ Now correctly aligns spatial and temporal features at location level
- ✅ Uses all CapsNet data (not discarding 91%)
- ✅ Mean pooling preserves spatial information while matching LSTM granularity
- ✅ Proper implementation of concatenation: `(n_samples, d_spatial + d_temporal)`

**Alternative Aggregation Strategies** (if needed later):
- Max pooling: `agg_dict = {col: 'max' for col in caps_feature_cols}`
- Median: `agg_dict = {col: 'median' for col in caps_feature_cols}`
- Keep as-is (mean): Best for general feature representation

### 📊 Complete Workflow: From Model Training to Final Predictions

Here's the **complete step-by-step flow** after models are trained:

---

#### **STEP 1: Feature Extraction (CapsNet) 🖼️**
**Location:** `extract_capsnet_features_fold()` function

```
Trained CapsNet Model (.pth file)
        ↓
Load model checkpoint
        ↓
Load validation data (fold-specific)
        ↓
Pass images through model in batches
        ↓
Extract 64-dimensional features per patch
        ↓
Store metadata (lat, lon, pm25) from each patch
        ↓
Save to CSV: capsnet_features_fold_X.csv
```

**Output CSV Structure:**
```
caps_f_0, caps_f_1, ..., caps_f_63, pm25, lat, lon
    0.23,    -0.15, ...,     0.82,   45,  14.5, 121.0  ← Patch 1 of Location A
   -0.11,     0.44, ...,     0.19,   45,  14.5, 121.0  ← Patch 2 of Location A
    0.55,    -0.22, ...,     0.33,   45,  14.5, 121.0  ← Patch 3 of Location A
    ...
    0.12,     0.88, ...,    -0.44,   62,  14.6, 121.1  ← Patch 1 of Location B
```
**Result:** 84,770 rows (10 patches × ~8,477 locations)

---

#### **STEP 2: Feature Extraction (LSTM) ⏱️**
**Location:** `extract_lstm_features_fold()` function

```
Trained LSTM Model (.pth file)
        ↓
Load temporal data (60-timestep sequences)
        ↓
Load validation data (fold-specific)
        ↓
Pass sequences through model
        ↓
Extract 32-dimensional features per location
        ↓
Save to CSV: lstm_features_fold_X.csv
```

**Output CSV Structure:**
```
lstm_f_0, lstm_f_1, ..., lstm_f_31
    0.45,    -0.23, ...,     0.67    ← Location-timestamp 1
   -0.12,     0.55, ...,     0.34    ← Location-timestamp 2
    0.78,    -0.09, ...,    -0.21    ← Location-timestamp 3
    ...
```
**Result:** 7,694 rows (one per location-timestamp)

---

#### **STEP 3: Feature Fusion (Patch Aggregation) ⚡**
**Location:** `fuse_features_fold()` function - **THIS IS WHERE AGGREGATION HAPPENS!**

```
CapsNet CSV (84,770 patches)         LSTM CSV (7,694 locations)
        ↓                                        ↓
    Read CSV                                 Read CSV
        ↓                                        ↓
Check if 'lat' & 'lon' exist                    |
        ↓                                        |
    YES? ✅                                      |
        ↓                                        |
┌───────────────────────────────────┐            |
│  GROUP BY (lat, lon)              │            |
│  AGGREGATE using MEAN             │            |
│                                   │            |
│  caps_f_0 = mean(all patch caps_f_0)          |
│  caps_f_1 = mean(all patch caps_f_1)          |
│  ...                              │            |
│  caps_f_63 = mean(all patch caps_f_63)        |
│  pm25 = first(pm25)               │            |
└───────────────────────────────────┘            |
        ↓                                        |
Aggregated: 7,694 locations                     |
        ↓                                        ↓
    ┌────────────────────────────────────────────┐
    │      CONCATENATE HORIZONTALLY              │
    │  (CapsNet 64D + LSTM 32D = 96D total)     │
    └────────────────────────────────────────────┘
                        ↓
            Save to CSV: fused_features_fold_X.csv
```

**Aggregation Example:**
```python
# BEFORE aggregation (84,770 rows):
Location A: Patch 1 → [0.23, -0.15, ..., 0.82]  (64 features)
Location A: Patch 2 → [-0.11, 0.44, ..., 0.19]  (64 features)
Location A: Patch 3 → [0.55, -0.22, ..., 0.33]  (64 features)
... (10 patches total)

# AFTER aggregation (1 row for Location A):
Location A: MEAN → [0.22, 0.02, ..., 0.45]  (64 features averaged)
```

**Fused CSV Structure:**
```
caps_f_0, ..., caps_f_63, lstm_f_0, ..., lstm_f_31, pm2.5
   0.22,  ...,     0.45,      0.45,  ...,     0.67,   45   ← Location 1
   0.33,  ...,     0.28,     -0.12,  ...,     0.34,   52   ← Location 2
   ...
```
**Result:** 7,694 rows × 97 columns (64 CapsNet + 32 LSTM + 1 target)

---

#### **STEP 4: LightGBM Training 🌳**
**Location:** `train_lightgbm_fold()` function

```
Fused Features CSV (7,694 × 97)
        ↓
Separate X (96 features) and y (pm2.5 target)
        ↓
Split into train/val (80/20)
        ↓
Train gradient boosting trees
        ↓
Make predictions on validation set
        ↓
Calculate metrics (RMSE, MAE, R²)
        ↓
Save model: lightgbm_fold_X.txt
```

**Result:** Trained LightGBM model + predictions + metrics

---

#### **Key Takeaway:**
- ✅ **Aggregation happens in STEP 3** (`fuse_features_fold()`)
- ✅ **Method:** Group by `(lat, lon)` and take mean of all patch features
- ✅ **Reduces:** 84,770 patches → 7,694 locations
- ✅ **Then:** Concatenates with 7,694 LSTM features
- ✅ **Result:** Aligned (7,694 × 96) feature matrix for LightGBM

### 🔍 Detailed View: Patch Aggregation Mechanism

**What happens inside `fuse_features_fold()` line-by-line:**

```python
# 1. Check if location columns exist
has_location = 'lat' in capsnet_df.columns and 'lon' in capsnet_df.columns
# Result: True ✅ (we saved lat/lon during extraction)

# 2. Get all feature column names
caps_feature_cols = [c for c in capsnet_df.columns if c.startswith('caps_f_')]
# Result: ['caps_f_0', 'caps_f_1', ..., 'caps_f_63']  (64 columns)

# 3. Define aggregation strategy
agg_dict = {col: 'mean' for col in caps_feature_cols}
agg_dict['pm25'] = 'first'
# Result: {
#   'caps_f_0': 'mean',
#   'caps_f_1': 'mean',
#   ...
#   'caps_f_63': 'mean',
#   'pm25': 'first'
# }

# 4. GROUP BY location and AGGREGATE
capsnet_aggregated = capsnet_df.groupby(['lat', 'lon'], as_index=False).agg(agg_dict)
# This is THE MAGIC LINE! 🪄
```

**What `groupby(['lat', 'lon']).agg(agg_dict)` does:**

```
BEFORE (84,770 rows):
┌─────────┬────────┬──────────┬──────────┬─────┬──────────┬──────┐
│   lat   │  lon   │ caps_f_0 │ caps_f_1 │ ... │ caps_f_63│ pm25 │
├─────────┼────────┼──────────┼──────────┼─────┼──────────┼──────┤
│  14.500 │ 121.00 │   0.23   │  -0.15   │ ... │   0.82   │  45  │ ← Patch 1
│  14.500 │ 121.00 │  -0.11   │   0.44   │ ... │   0.19   │  45  │ ← Patch 2
│  14.500 │ 121.00 │   0.55   │  -0.22   │ ... │   0.33   │  45  │ ← Patch 3
│  14.500 │ 121.00 │   0.08   │   0.31   │ ... │  -0.12   │  45  │ ← Patch 4
│    ...  │   ...  │   ...    │   ...    │ ... │   ...    │  ... │
│  14.500 │ 121.00 │  -0.33   │   0.19   │ ... │   0.44   │  45  │ ← Patch 10
│─────────┼────────┼──────────┼──────────┼─────┼──────────┼──────│
│  14.501 │ 121.01 │   0.67   │  -0.44   │ ... │   0.23   │  52  │ ← Next location
│    ...  │   ...  │   ...    │   ...    │ ... │   ...    │  ... │
└─────────┴────────┴──────────┴──────────┴─────┴──────────┴──────┘

                              ↓
                    [GROUPBY + MEAN]
                              ↓

AFTER (7,694 rows):
┌─────────┬────────┬──────────┬──────────┬─────┬──────────┬──────┐
│   lat   │  lon   │ caps_f_0 │ caps_f_1 │ ... │ caps_f_63│ pm25 │
├─────────┼────────┼──────────┼──────────┼─────┼──────────┼──────┤
│  14.500 │ 121.00 │   0.22   │   0.02   │ ... │   0.45   │  45  │ ← Location 1 (mean of 10 patches)
│  14.501 │ 121.01 │   0.33   │  -0.28   │ ... │   0.18   │  52  │ ← Location 2 (mean of 10 patches)
│    ...  │   ...  │   ...    │   ...    │ ... │   ...    │  ... │
└─────────┴────────┴──────────┴──────────┴─────┴──────────┴──────┘
```

**Then concatenate with LSTM:**
```
CapsNet Aggregated (7,694 × 66)    LSTM Features (7,694 × 32)
┌──────────────────────────┐      ┌──────────────────────┐
│ caps_f_0 ... caps_f_63   │      │ lstm_f_0 ... lstm_f_31│
│   0.22         0.45       │  +   │   0.45        0.67    │
│   0.33         0.18       │      │  -0.12        0.34    │
│   ...          ...        │      │   ...         ...     │
└──────────────────────────┘      └──────────────────────┘
                    ↓
            [pd.concat axis=1]
                    ↓
        Fused Features (7,694 × 97)
┌──────────────────────────────────────────────────────┐
│ caps_f_0 ... caps_f_63  lstm_f_0 ... lstm_f_31  pm25 │
│   0.22        0.45        0.45         0.67       45  │
│   0.33        0.18       -0.12         0.34       52  │
│   ...         ...         ...          ...       ... │
└──────────────────────────────────────────────────────┘
                    ↓
              LightGBM Training
```

**Summary:**
- **WHERE:** Aggregation happens in `fuse_features_fold()` function
- **WHEN:** After both CapsNet and LSTM features are extracted
- **HOW:** Using pandas `groupby(['lat', 'lon']).agg({'caps_f_*': 'mean'})`
- **WHY:** To align patch-level (84,770) with location-level (7,694) features

In [ ]:
def fuse_features_and_train_lightgbm(self) -> Dict[int, Dict]:
    """Fuse features from all folds and train LightGBM"""
    print(f"\nFusing features and training LightGBM for all folds...")
    
    fold_results = {}
    
    for fold in range(1, self.n_folds + 1):
        print(f"\nProcessing Fold {fold}")
        print("-" * 40)
        
        # Load features for this fold
        capsnet_df = self.capsnet_features[fold]
        lstm_df = self.lstm_features[fold]
        
        # Fuse features
        print("   Fusing CapsNet and LSTM features...")
        fused_features = self.fuse_features_fold(capsnet_df, lstm_df, fold)
        
        # Train LightGBM
        print("   Training LightGBM...")
        fold_result = self.train_lightgbm_fold(fused_features, fold)
        fold_results[fold] = fold_result
        
        print(f"   Fold {fold} LightGBM training completed!")
        print(f"       RMSE: {fold_result['rmse']:.4f}")
        print(f"       MAE: {fold_result['mae']:.4f}")  
        print(f"       R²: {fold_result['r2']:.4f}")
    
    self.fold_results = fold_results
    print(f"\nAll LightGBM models trained!")
    return fold_results

def fuse_features_fold(self, capsnet_df: pd.DataFrame, lstm_df: pd.DataFrame, 
                      fold: int) -> pd.DataFrame:
    """Fuse CapsNet and LSTM features for a specific fold
    
    This function properly aligns patch-level CapsNet features with location-level LSTM features
    by aggregating multiple patches per location using mean pooling.
    """
    
    # Check if CapsNet has location metadata (lat/lon) for proper alignment
    has_location = 'lat' in capsnet_df.columns and 'lon' in capsnet_df.columns
    
    if has_location:
        print(f"       Aggregating {len(capsnet_df)} CapsNet patches by location...")
        
        # Get all CapsNet feature columns
        caps_feature_cols = [c for c in capsnet_df.columns if c.startswith('caps_f_')]
        
        # Group by location (lat, lon) and aggregate patches using mean
        # This reduces 84,770 patches → ~7,694 locations
        agg_dict = {col: 'mean' for col in caps_feature_cols}
        agg_dict['pm25'] = 'first'  # PM2.5 is the same for all patches at a location
        
        capsnet_aggregated = capsnet_df.groupby(['lat', 'lon'], as_index=False).agg(agg_dict)
        
        print(f"          Aggregated to {len(capsnet_aggregated)} unique locations")
        print(f"          LSTM has {len(lstm_df)} samples")
        
        # Now both should have similar lengths (location-level)
        # Align by row index (assumes both are sorted by same location order)
        min_len = min(len(capsnet_aggregated), len(lstm_df))
        capsnet_features = capsnet_aggregated.iloc[:min_len]
        lstm_features = lstm_df.iloc[:min_len]
        
    else:
        # Fallback: No location metadata available
        print(f"       WARNING: No location metadata found in CapsNet features!")
        print(f"       Falling back to simple truncation (not recommended for production)")
        min_len = min(len(capsnet_df), len(lstm_df))
        capsnet_features = capsnet_df.iloc[:min_len]
        lstm_features = lstm_df.iloc[:min_len]
    
    # Combine features
    fused_df = pd.concat([
        capsnet_features.reset_index(drop=True),
        lstm_features.reset_index(drop=True)
    ], axis=1)
    
    # Handle duplicate pm25 columns
    if 'pm25' in capsnet_features.columns:
        # Use CapsNet's pm25 (already in fused_df)
        pass
    
    # Rename for clarity
    fused_df.rename(columns={'pm25': 'pm2.5'}, inplace=True, errors='ignore')
    
    # Save fused features
    fused_path = f"{self.output_dir}/features/fused/fused_features_fold_{fold}_{self.day_folder}.csv"
    fused_df.to_csv(fused_path, index=False)
    
    # Count features
    caps_feat_count = len([c for c in fused_df.columns if c.startswith('caps_f_')])
    lstm_feat_count = len([c for c in fused_df.columns if c.startswith('lstm_f_')])
    
    print(f"       Fused features shape: {fused_df.shape}")
    print(f"       CapsNet features: {caps_feat_count}")
    print(f"       LSTM features: {lstm_feat_count}")
    print(f"       Total features: {caps_feat_count + lstm_feat_count}")
    
    return fused_df

def train_lightgbm_fold(self, fused_df: pd.DataFrame, fold: int) -> Dict:
    """Train LightGBM for a specific fold"""
    
    # Prepare features and target
    feature_cols = [c for c in fused_df.columns if c.startswith(('caps_f_', 'lstm_f_'))]
    
    # Handle different possible target column names
    if 'pm2.5' in fused_df.columns:
        target_col = 'pm2.5'
    elif 'pm25' in fused_df.columns:
        target_col = 'pm25'
    else:
        raise ValueError(f"No PM2.5 target column found! Available columns: {fused_df.columns.tolist()}")
    
    X = fused_df[feature_cols]
    y = fused_df[target_col]
    
    # Remove any NaN values
    mask = ~(X.isna().any(axis=1) | y.isna())
    X = X[mask]
    y = y[mask]
    
    print(f"       Training samples: {len(X)}")
    print(f"       Feature columns: {len(feature_cols)}")
    
    # Split for training/validation within fold
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # LightGBM parameters (optimized for fast mode)
    if self.fast_mode:
        # Fast mode: fewer iterations, simpler model
        lgb_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'num_leaves': 15,  # Reduced from 31
            'learning_rate': 0.1,  # Increased from 0.05
            'feature_fraction': 0.8,
            'bagging_fraction': 0.7,
            'bagging_freq': 5,
            'verbose': -1,
            'random_state': 42,
            'n_jobs': -1  # Use all CPU cores
        }
        num_boost_round = 100  # Reduced from 1000
        early_stopping_rounds = 20  # Reduced from 50
    else:
        # Full mode: more iterations, complex model
        lgb_params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'num_leaves': 31,
            'learning_rate': 0.05,
            'feature_fraction': 0.9,
            'bagging_fraction': 0.8,
            'bagging_freq': 5,
            'verbose': -1,
            'random_state': 42,
            'n_jobs': -1
        }
        num_boost_round = 1000
        early_stopping_rounds = 50
    
    # Create datasets
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)
    
    # Train model
    model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        num_boost_round=num_boost_round,
        callbacks=[lgb.early_stopping(early_stopping_rounds), lgb.log_evaluation(0)]
    )
    
    # Make predictions
    y_pred = model.predict(X_val)
    
    # Calculate metrics
    n_samples = len(X_val)
    n_features = len(feature_cols)
    
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    mae = mean_absolute_error(y_val, y_pred)
    r2 = r2_score(y_val, y_pred)
    
    # Calculate adjusted R-squared
    if n_samples > n_features + 1:
        adj_r2 = 1 - ((1 - r2) * (n_samples - 1) / (n_samples - n_features - 1))
    else:
        adj_r2 = r2
    
    # Save model
    model_path = f"{self.output_dir}/models/lightgbm/lightgbm_fold_{fold}_{self.day_folder}.txt"
    model.save_model(model_path)
    
    mode_text = "FAST" if self.fast_mode else "FULL"
    print(f"       LightGBM trained ({mode_text} mode: {model.num_trees()} trees)")
    
    return {
        'fold': fold,
        'model_path': model_path,
        'rmse': rmse,
        'mae': mae,
        'r2': r2,
        'adj_r2': adj_r2,
        'feature_importance': dict(zip(feature_cols, model.feature_importance())),
        'predictions': y_pred,
        'actuals': y_val.values
    }

# Add methods to the class
CompleteMLPipeline.fuse_features_and_train_lightgbm = fuse_features_and_train_lightgbm
CompleteMLPipeline.fuse_features_fold = fuse_features_fold
CompleteMLPipeline.train_lightgbm_fold = train_lightgbm_fold
print("Feature fusion and LightGBM training methods added (with PROPER PATCH AGGREGATION)!")

## 8. Results Analysis and Metrics

Analyze cross-validation results across all folds and calculate comprehensive metrics.

In [ ]:
def analyze_results(self) -> Dict:
    """Analyze and compare results across all folds"""
    print(f"\nAnalyzing results across all {self.n_folds} folds...")
    
    # Collect metrics
    fold_metrics = []
    for fold, result in self.fold_results.items():
        fold_metrics.append({
            'fold': fold,
            'rmse': result['rmse'],
            'mae': result['mae'],
            'r2': result['r2'],
            'adj_r2': result['adj_r2']
        })
    
    metrics_df = pd.DataFrame(fold_metrics)
    
    # Calculate statistics
    stats = {
        'mean_rmse': metrics_df['rmse'].mean(),
        'std_rmse': metrics_df['rmse'].std(),
        'mean_mae': metrics_df['mae'].mean(),
        'std_mae': metrics_df['mae'].std(),
        'mean_r2': metrics_df['r2'].mean(),
        'std_r2': metrics_df['r2'].std(),
        'mean_adj_r2': metrics_df['adj_r2'].mean(),
        'std_adj_r2': metrics_df['adj_r2'].std(),
        'best_fold': metrics_df.loc[metrics_df['rmse'].idxmin(), 'fold'],
        'worst_fold': metrics_df.loc[metrics_df['rmse'].idxmax(), 'fold']
    }
    
    self.final_results = {
        'fold_metrics': fold_metrics,
        'statistics': stats,
        'day_folder': self.day_folder
    }
    
    # Print results
    print(f"\nCross-Validation Results Summary:")
    print(f"   Average RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Average MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Average R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    print(f"   Average Adjusted R²: {stats['mean_adj_r2']:.4f} ± {stats['std_adj_r2']:.4f}")
    print(f"   Best fold:    {stats['best_fold']} (RMSE: {metrics_df.loc[stats['best_fold']-1, 'rmse']:.4f})")
    print(f"   Worst fold:   {stats['worst_fold']} (RMSE: {metrics_df.loc[stats['worst_fold']-1, 'rmse']:.4f})")
    
    # Save results
    results_path = f"{self.output_dir}/results/cv_results_{self.day_folder}.json"
    with open(results_path, 'w') as f:
        json.dump(self.final_results, f, indent=2, default=str)
    
    metrics_path = f"{self.output_dir}/results/fold_metrics_{self.day_folder}.csv"
    metrics_df.to_csv(metrics_path, index=False)
    
    print(f"   Results saved to: {results_path}")
    print(f"   Metrics saved to: {metrics_path}")
    
    return self.final_results

# Add the method to the class
CompleteMLPipeline.analyze_results = analyze_results
print("Results analysis method added!")

## 9. Visualization Creation

Create comprehensive visualizations for model evaluation and results interpretation.

In [ ]:
def create_visualizations(self):
    """Create visualizations for the results"""
    print(f"\nCreating visualizations...")
    
    # 1. Fold comparison plot
    self.plot_fold_comparison()
    
    # 2. Feature importance plot
    self.plot_feature_importance()
    
    # 3. Predictions vs actual plot
    self.plot_predictions_vs_actual()
    
    print(f"   Visualizations saved to: {self.output_dir}/plots/")

def plot_fold_comparison(self):
    """Plot comparison of metrics across folds"""
    metrics_data = []
    for fold, result in self.fold_results.items():
        metrics_data.extend([
            {'fold': fold, 'metric': 'RMSE', 'value': result['rmse']},
            {'fold': fold, 'metric': 'MAE', 'value': result['mae']},
            {'fold': fold, 'metric': 'R²', 'value': result['r2']}
        ])
    
    metrics_df = pd.DataFrame(metrics_data)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    for i, metric in enumerate(['RMSE', 'MAE', 'R²']):
        data = metrics_df[metrics_df['metric'] == metric]
        axes[i].bar(data['fold'], data['value'], alpha=0.7)
        axes[i].set_title(f'{metric} by Fold')
        axes[i].set_xlabel('Fold')
        axes[i].set_ylabel(metric)
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/fold_comparison_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_feature_importance(self):
    """Plot feature importance across folds"""
    # Aggregate feature importance across folds
    all_importance = {}
    for fold, result in self.fold_results.items():
        for feature, importance in result['feature_importance'].items():
            if feature not in all_importance:
                all_importance[feature] = []
            all_importance[feature].append(importance)
    
    # Calculate mean importance
    mean_importance = {k: np.mean(v) for k, v in all_importance.items()}
    
    # Sort by importance
    sorted_features = sorted(mean_importance.items(), key=lambda x: x[1], reverse=True)
    
    # Plot top 20 features
    top_features = sorted_features[:20]
    features, importance = zip(*top_features)
    
    plt.figure(figsize=(12, 8))
    plt.barh(range(len(features)), importance, alpha=0.7)
    plt.yticks(range(len(features)), features)
    plt.xlabel('Feature Importance')
    plt.title(f'Top 20 Feature Importance - {self.day_folder}')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/feature_importance_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

def plot_predictions_vs_actual(self):
    """Plot predictions vs actual values for all folds"""
    n_cols = 3
    n_rows = (self.n_folds + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
    
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    for fold, result in self.fold_results.items():
        row = (fold - 1) // n_cols
        col = (fold - 1) % n_cols
        ax = axes[row, col]
        
        actual = result['actual']
        pred = result['predictions']
        
        # Scatter plot
        ax.scatter(actual, pred, alpha=0.6)
        
        # Perfect prediction line
        min_val = min(actual.min(), pred.min())
        max_val = max(actual.max(), pred.max())
        ax.plot([min_val, max_val], [min_val, max_val], 'r--', alpha=0.8)
        
        ax.set_xlabel('Actual PM2.5')
        ax.set_ylabel('Predicted PM2.5')
        ax.set_title(f'Fold {fold} - R² = {result["r2"]:.4f}')
        ax.grid(True, alpha=0.3)
    
    # Remove empty subplots if any
    for i in range(self.n_folds, n_rows * n_cols):
        row = i // n_cols
        col = i % n_cols
        fig.delaxes(axes[row, col])
    
    plt.tight_layout()
    plt.savefig(f"{self.output_dir}/plots/predictions_vs_actual_{self.day_folder}.png", dpi=300, bbox_inches='tight')
    plt.show()

# Add the methods to the class
CompleteMLPipeline.create_visualizations = create_visualizations
CompleteMLPipeline.plot_fold_comparison = plot_fold_comparison
CompleteMLPipeline.plot_feature_importance = plot_feature_importance
CompleteMLPipeline.plot_predictions_vs_actual = plot_predictions_vs_actual
print("Visualization methods added!")

## 10. Main Pipeline Execution

Complete pipeline execution method that orchestrates all the components.

## CRITICAL FIX: Proper Per-Fold Execution

### Previous Implementation (WRONG):
```
Step 2: Train ALL LSTM folds (1-5)
        Train ALL CapsNet folds (1-5)
Step 3: Extract features from ALL folds
Step 4: Fuse and train LightGBM for ALL folds
```

### Current Implementation (CORRECT):
```
For Fold 1:
  → Train CapsNet for Fold 1
  → Train LSTM for Fold 1
  → Extract CapsNet features for Fold 1
  → Extract LSTM features for Fold 1
  → Fuse features for Fold 1
  → Train LightGBM for Fold 1
  → Evaluate Fold 1

For Fold 2:
  → Train CapsNet for Fold 2
  → Train LSTM for Fold 2
  → Extract CapsNet features for Fold 2
  → Extract LSTM features for Fold 2
  → Fuse features for Fold 2
  → Train LightGBM for Fold 2
  → Evaluate Fold 2

... (repeat for all 5 folds)
```

### Why This Matters:
- **Standard CV Practice**: Each fold is completely independent
- **Memory Efficiency**: Only one fold's models in memory at a time
- **Proper Evaluation**: Each fold evaluated immediately after training
- **Thesis Compliance**: Matches standard hybrid model methodology
- **Debugging**: Easier to identify issues in specific folds

In [ ]:
def run_complete_pipeline(self) -> Dict:
    """Run the complete pipeline with proper per-fold execution"""
    print(f"Starting Complete ML Pipeline for {self.day_folder}")
    print("=" * 60)
    print(" CRITICAL: Per-Fold Execution")
    print("   For EACH fold: Train LSTM → Train CapsNet → Extract LSTM features → Extract CapsNet features → Fuse → Train LightGBM → Evaluate")
    print("=" * 60)
    
    try:
        # Step 1: Load hyperparameters
        print(f"\nStep 1: Loading Best Hyperparameters")
        capsnet_params = self.load_best_hyperparameters('capsnet')
        lstm_params = self.load_best_hyperparameters('lstm')
        
        # Step 2: Execute complete workflow for EACH fold
        print(f"\nStep 2: Executing {self.n_folds}-Fold Cross-Validation")
        print("=" * 60)
        
        fold_results = {}
        
        for fold in range(1, self.n_folds + 1):
            print(f"\n{'='*70}")
            print(f"FOLD {fold}/{self.n_folds} - COMPLETE WORKFLOW")
            print(f"{'='*70}")
            
            # 2a: Train LSTM for this fold
            print(f"\n   Step {fold}.1: Training LSTM for Fold {fold}")
            lstm_model_path = self.train_lstm_fold(lstm_params, fold)
            
            # 2b: Train CapsNet for this fold
            print(f"\n   Step {fold}.2: Training CapsNet for Fold {fold}")
            capsnet_model_path = self.train_capsnet_fold(capsnet_params, fold)
            
            # 2c: Extract LSTM features for this fold
            print(f"\n   Step {fold}.3: Extracting LSTM Features for Fold {fold}")
            lstm_features = self.extract_lstm_features_fold(lstm_model_path, fold, lstm_params)
            
            # 2d: Extract CapsNet features for this fold
            print(f"\n   Step {fold}.4: Extracting CapsNet Features for Fold {fold}")
            capsnet_features = self.extract_capsnet_features_fold(capsnet_model_path, fold)
            
            # 2e: Fuse features for this fold
            print(f"\n   Step {fold}.5: Fusing Features for Fold {fold}")
            fused_features = self.fuse_features_fold(capsnet_features, lstm_features, fold)
            
            # 2f: Train LightGBM for this fold
            print(f"\n   Step {fold}.6: Training LightGBM for Fold {fold}")
            fold_result = self.train_lightgbm_fold(fused_features, fold)
            fold_results[fold] = fold_result
            
            # Print fold summary
            print(f"\n   FOLD {fold} COMPLETE!")
            print(f"      RMSE: {fold_result['rmse']:.4f}")
            print(f"      MAE: {fold_result['mae']:.4f}")
            print(f"      R²: {fold_result['r2']:.4f}")
            print(f"      Adjusted R²: {fold_result['adj_r2']:.4f}")
            print(f"{'='*70}\n")
        
        # Store fold results
        self.fold_results = fold_results
        
        # Step 3: Analyze results across all folds
        print(f"\nStep 3: Analyzing Results Across All Folds")
        results = self.analyze_results()
        
        # Step 4: Create visualizations
        print(f"\nStep 4: Creating Visualizations")
        self.create_visualizations()
        
        print(f"\nComplete Pipeline Finished Successfully!")
        print("=" * 60)
        
        return results
        
    except Exception as e:
        print(f"\nPipeline failed: {e}")
        traceback.print_exc()
        return None

# Add the method to the class
CompleteMLPipeline.run_complete_pipeline = run_complete_pipeline
print("Main pipeline execution method updated with proper per-fold workflow!")

## 11. Cross-Day Comparison

Functions for running the pipeline across multiple days and creating comparative analysis.

In [ ]:
def create_cross_day_comparison(all_results: Dict, output_dir: str):
    """Create comparison plots across different days"""
    comparison_data = []
    
    for day, result in all_results.items():
        if result and 'statistics' in result:
            stats = result['statistics']
            comparison_data.append({
                'day': day,
                'mean_rmse': stats['mean_rmse'],
                'std_rmse': stats['std_rmse'],
                'mean_mae': stats['mean_mae'],
                'std_mae': stats['std_mae'],
                'mean_r2': stats['mean_r2'],
                'std_r2': stats['std_r2']
            })
    
    if comparison_data:
        comparison_df = pd.DataFrame(comparison_data)
        
        # Create comparison plots
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        for i, metric in enumerate(['rmse', 'mae', 'r2']):
            mean_col = f'mean_{metric}'
            std_col = f'std_{metric}'
            
            axes[i].bar(comparison_df['day'], comparison_df[mean_col], 
                       yerr=comparison_df[std_col], alpha=0.7, capsize=5)
            axes[i].set_title(f'{metric.upper()} Comparison Across Days')
            axes[i].set_ylabel(metric.upper())
            axes[i].tick_params(axis='x', rotation=45)
            axes[i].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{output_dir}/cross_day_comparison.png", dpi=300, bbox_inches='tight')
        plt.show()
        
        # Save comparison data
        comparison_df.to_csv(f"{output_dir}/cross_day_results.csv", index=False)
        
        print(f"Cross-day comparison saved to: {output_dir}/")
        return comparison_df
    
    return None

def run_pipeline_for_all_days(output_base_dir: str = "pipeline_outputs", fast_mode: bool = False):
    """Run pipeline for all available days"""
    print("Running Complete Pipeline for All Days")
    print("=" * 60)
    
    days = ['7_24_data', '10_19_data', '11_10_data']
    all_results = {}
    
    for day in days:
        print(f"\nProcessing {day}...")
        pipeline = CompleteMLPipeline(day, f"{output_base_dir}/{day}", fast_mode=fast_mode)
        result = pipeline.run_complete_pipeline()
        all_results[day] = result
    
    # Create comparison across days
    print(f"\nCreating Cross-Day Comparison...")
    comparison_df = create_cross_day_comparison(all_results, output_base_dir)
    
    return all_results, comparison_df

print("Cross-day comparison functions defined!")

## 12. Interactive Pipeline Execution

Now you can run the pipeline interactively! Choose your configuration and execute.

In [ ]:
# Configuration
DAY_FOLDER = '7_24_data'  # Change this to: '7_24_data', '10_19_data', or '11_10_data'
OUTPUT_DIR = 'pipeline_outputs'
FAST_MODE = True  # Set to False for full 5-fold CV, True for ultra-fast 2-fold testing

print(f"Configuration:")
print(f"   Day folder: {DAY_FOLDER}")
print(f"   Output directory: {OUTPUT_DIR}")
print(f"   Fast mode: {'ON (2-fold CV for ultra-fast testing)' if FAST_MODE else 'OFF (5-fold CV for full evaluation)'}")
print(f"   Cross-validation folds: {2 if FAST_MODE else 5}")

if FAST_MODE:
    print(f"\n⚡ ULTRA-FAST MODE OPTIMIZATIONS:")
    print(f"   • CV Folds: 2 instead of 5 (60% time reduction)")
    print(f"   • CapsNet epochs: 1 instead of 10 (90% time reduction)")
    print(f"   • LSTM epochs: 1 instead of 10 (90% time reduction)")
    print(f"   • CapsNet batch size: 16 (larger for faster extraction)")
    print(f"   • LSTM batch size: 64 (larger for faster extraction)")
    print(f"   • LightGBM trees: max 100 instead of 1000")
    print(f"   • LightGBM learning rate: 0.1 instead of 0.05")
    print(f"   • LightGBM early stopping: 20 rounds instead of 50")
    print(f"   • Estimated time: ~10-15 minutes per dataset ⚡⚡⚡")
    print(f"   • Use case: Quick validation, debugging, code testing")
else:
    print(f"\n🎯 FULL MODE (Production Quality):")
    print(f"   • CV Folds: 5 for robust evaluation")
    print(f"   • CapsNet epochs: 10")
    print(f"   • LSTM epochs: 10")
    print(f"   • CapsNet batch size: 8 (standard)")
    print(f"   • LSTM batch size: 32 (standard)")
    print(f"   • LightGBM trees: max 1000 with early stopping")
    print(f"   • LightGBM learning rate: 0.05 (more careful)")
    print(f"   • LightGBM early stopping: 50 rounds")
    print(f"   • Estimated time: ~2-3 hours per dataset")
    print(f"   • Use case: Final results, thesis submission")

In [ ]:
# Initialize and run the pipeline for a single day
print(f"\nInitializing Complete ML Pipeline...")

pipeline = CompleteMLPipeline(
    day_folder=DAY_FOLDER,
    output_dir=OUTPUT_DIR,
    fast_mode=FAST_MODE
)

print(f"\nPipeline initialized successfully!")
print(f"   Ready to train {pipeline.n_folds}-fold cross-validation")

In [ ]:
# Run the complete pipeline
# This cell will execute the entire pipeline - may take several hours depending on configuration

print("Starting Complete Pipeline Execution...")
print("This may take several hours depending on your configuration")
print("You can monitor progress in the output below")

results = pipeline.run_complete_pipeline()

In [ ]:
# Alternative: Run pipeline for all days (if you want to compare across days)
# This will take significantly longer as it processes all three datasets

print("Option: Run Pipeline for All Days")
print("This will take much longer as it processes all datasets")
print("Only run this if you want cross-day comparison")

# Uncomment the lines below to run for all days
# all_results, comparison_df = run_pipeline_for_all_days(
#     output_base_dir="complete_pipeline_outputs",
#     fast_mode=FAST_MODE
# )

print("Uncomment the lines above to run for all days")
print("This will process: 7_24_data, 10_19_data, and 11_10_data")

## 13. Results Inspection

After running the pipeline, use these cells to inspect and analyze the results.

In [ ]:
# Inspect pipeline results (run this after the pipeline completes)
# This cell will display the final results and statistics

if 'results' in locals() and results is not None:
    print("Pipeline Results Summary:")
    print("=" * 50)
    
    stats = results['statistics']
    print(f"Cross-Validation Statistics for {results['day_folder']}:")
    print(f"   Mean RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Mean MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Mean R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    print(f"   Best Fold: {stats['best_fold']}")
    print(f"   Worst Fold: {stats['worst_fold']}")
    
    # Display fold metrics
    fold_metrics_df = pd.DataFrame(results['fold_metrics'])
    print(f"\nIndividual Fold Performance:")
    print(fold_metrics_df.round(4))
    
else:
    print("No results found. Please run the pipeline first.")
    print("Make sure to uncomment the execution line in the previous cell")

In [ ]:
# Load and display saved results (if you want to examine results from a previous run)
import glob
import json

# Look for saved results
result_files = glob.glob(f"{OUTPUT_DIR}/results/cv_results_*.json")

if result_files:
    print(f"Found {len(result_files)} result files:")
    for file in result_files:
        print(f"   - {file}")
    
    # Load the most recent results
    latest_file = max(result_files, key=os.path.getmtime)
    print(f"\nLoading results from: {latest_file}")
    
    with open(latest_file, 'r') as f:
        saved_results = json.load(f)
    
    # Display summary
    stats = saved_results['statistics']
    print(f"\nSaved Results Summary for {saved_results['day_folder']}:")
    print(f"   Mean RMSE: {stats['mean_rmse']:.4f} ± {stats['std_rmse']:.4f}")
    print(f"   Mean MAE:  {stats['mean_mae']:.4f} ± {stats['std_mae']:.4f}")
    print(f"   Mean R²:   {stats['mean_r2']:.4f} ± {stats['std_r2']:.4f}")
    
else:
    print("No saved results found.")
    print("Run the pipeline first to generate results")

## Notes and Next Steps

**What this notebook does:**
1. Loads best hyperparameters from previous tuning
2. Trains CapsNet and LSTM with proper K-fold cross-validation
3. Extracts features from all trained models
4. Fuses CapsNet and LSTM features intelligently
5. Trains LightGBM on fused features
6. Provides comprehensive evaluation metrics
7. Creates publication-ready visualizations
8. Supports cross-day comparison analysis

**Key Features:**
- **Proper Cross-Validation**: No data leakage between folds
- **Fast Mode**: 3-fold CV for quick testing
- **Comprehensive Metrics**: RMSE, MAE, R² with confidence intervals
- **Rich Visualizations**: Fold comparison, feature importance, predictions vs actual
- **Result Persistence**: All results saved to disk
- **Multi-Day Support**: Compare performance across different datasets

**Before Running:**
1. Ensure all data preprocessing is complete
2. Verify CapsNet and LSTM models are available
3. Check that hyperparameter tuning results exist
4. Confirm sufficient disk space for outputs

**After Running:**
1. Examine cross-validation statistics
2. Review feature importance plots
3. Analyze prediction quality across folds
4. Compare results across different days if applicable

**Configuration Tips:**
- Use `FAST_MODE=True` for initial testing (3-fold CV)
- Use `FAST_MODE=False` for final results (5-fold CV)
- Adjust `DAY_FOLDER` to process different datasets
- Check `OUTPUT_DIR` for all generated files

---
*This notebook provides a complete end-to-end pipeline for multi-modal air quality prediction using deep learning and ensemble methods.*

## 10.1 Skip Training - Extract from Saved Models

If you've already trained models and want to skip straight to extraction, use this method!

In [ ]:
def run_from_saved_models(self, capsnet_model_pattern: str = None, lstm_model_pattern: str = None) -> Dict:
    """Skip training and run pipeline from saved models
    
    This is useful when you've already trained models and want to skip the time-consuming
    training phase and go straight to feature extraction and evaluation.
    
    Args:
        capsnet_model_pattern: Pattern to find CapsNet models (e.g., 'capsnet_fold_*_7_24_data.pth')
                              If None, will auto-detect from output_dir
        lstm_model_pattern: Pattern to find LSTM models (e.g., 'lstm_fold_*_7_24_data.pth')
                           If None, will auto-detect from output_dir
    
    Returns:
        Dictionary with complete pipeline results
    
    Example:
        pipeline = CompleteMLPipeline('7_24_data', 'pipeline_outputs', fast_mode=True)
        results = pipeline.run_from_saved_models()
    """
    print(f"Starting Pipeline from Saved Models for {self.day_folder}")
    print("=" * 60)
    print(" SKIPPING TRAINING - Loading from saved models")
    print("=" * 60)
    
    try:
        # Step 1: Load hyperparameters (for compatibility)
        print(f"\nStep 1: Loading Hyperparameters")
        try:
            capsnet_params = self.load_best_hyperparameters('capsnet')
            lstm_params = self.load_best_hyperparameters('lstm')
            print("   Hyperparameters loaded successfully!")
        except:
            print("   Warning: Could not load hyperparameters, using defaults")
            capsnet_params = {}
            lstm_params = {}
        
        # Step 2: Find saved models
        print(f"\nStep 2: Finding Saved Models")
        
        if capsnet_model_pattern is None:
            capsnet_model_pattern = f"capsnet*fold_*_{self.day_folder}.pth"
        if lstm_model_pattern is None:
            lstm_model_pattern = f"lstm_fold_*_{self.day_folder}.pth"
        
        import glob
        capsnet_models_found = sorted(glob.glob(f"{self.output_dir}/models/capsnet/{capsnet_model_pattern}"))
        lstm_models_found = sorted(glob.glob(f"{self.output_dir}/models/lstm/{lstm_model_pattern}"))
        
        print(f"   Found {len(capsnet_models_found)} CapsNet models")
        print(f"   Found {len(lstm_models_found)} LSTM models")
        
        if len(capsnet_models_found) == 0 or len(lstm_models_found) == 0:
            raise FileNotFoundError(
                f"Could not find saved models!\n"
                f"   CapsNet models: {capsnet_models_found}\n"
                f"   LSTM models: {lstm_models_found}\n"
                f"   Looking in: {self.output_dir}/models/\n"
                f"   Make sure you've trained models first or check the paths."
            )
        
        # Map models to folds
        capsnet_models = {}
        lstm_models = {}
        
        for i, (capsnet_path, lstm_path) in enumerate(zip(capsnet_models_found, lstm_models_found), 1):
            if i <= self.n_folds:
                capsnet_models[i] = capsnet_path
                lstm_models[i] = lstm_path
                print(f"   Fold {i}:")
                print(f"      CapsNet: {capsnet_path}")
                print(f"      LSTM: {lstm_path}")
        
        # Step 3: Extract features from saved models
        print(f"\nStep 3: Extracting Features from All Folds")
        print("=" * 60)
        
        fold_results = {}
        
        for fold in range(1, self.n_folds + 1):
            if fold not in capsnet_models or fold not in lstm_models:
                print(f"\n   Skipping Fold {fold} - models not found")
                continue
                
            print(f"\n{'='*70}")
            print(f"FOLD {fold}/{self.n_folds} - EXTRACTING FEATURES")
            print(f"{'='*70}")
            
            # Extract LSTM features
            print(f"\n   Step {fold}.1: Extracting LSTM Features for Fold {fold}")
            lstm_features = self.extract_lstm_features_fold(lstm_models[fold], fold, lstm_params)
            
            # Extract CapsNet features
            print(f"\n   Step {fold}.2: Extracting CapsNet Features for Fold {fold}")
            capsnet_features = self.extract_capsnet_features_fold(capsnet_models[fold], fold, capsnet_params)
            
            # Fuse features
            print(f"\n   Step {fold}.3: Fusing Features for Fold {fold}")
            fused_features = self.fuse_features_fold(capsnet_features, lstm_features, fold)
            
            # Train LightGBM
            print(f"\n   Step {fold}.4: Training LightGBM for Fold {fold}")
            fold_result = self.train_lightgbm_fold(fused_features, fold)
            fold_results[fold] = fold_result
            
            # Print fold summary
            print(f"\n   FOLD {fold} COMPLETE!")
            print(f"      RMSE: {fold_result['rmse']:.4f}")
            print(f"      MAE: {fold_result['mae']:.4f}")
            print(f"      R²: {fold_result['r2']:.4f}")
            print(f"      Adjusted R²: {fold_result['adj_r2']:.4f}")
            print(f"{'='*70}\n")
        
        # Store fold results
        self.fold_results = fold_results
        
        # Step 4: Analyze results
        print(f"\nStep 4: Analyzing Results Across All Folds")
        results = self.analyze_results()
        
        # Step 5: Create visualizations
        print(f"\nStep 5: Creating Visualizations")
        self.create_visualizations()
        
        print(f"\nPipeline from Saved Models Finished Successfully!")
        print("=" * 60)
        print(f"⚡ Time saved by skipping training!")
        
        return results
        
    except Exception as e:
        print(f"\nPipeline failed: {e}")
        traceback.print_exc()
        return None

# Add the method to the class
CompleteMLPipeline.run_from_saved_models = run_from_saved_models
print("Skip-training method added! Use pipeline.run_from_saved_models() to skip training.")

### Usage Example - Skip Training

Use this when you've already trained models and want to skip straight to extraction:

In [ ]:
# EXAMPLE: Skip training and use saved models
# Uncomment and run this instead of run_complete_pipeline()

pipeline = CompleteMLPipeline(DAY_FOLDER, OUTPUT_DIR, fast_mode=FAST_MODE)
results = pipeline.run_from_saved_models()

# This will:
# 1. Find all saved CapsNet and LSTM models
# 2. Extract features from validation sets
# 3. Fuse features
# 4. Train LightGBM (fast!)
# 5. Analyze and visualize results
# 
# ⚡ Saves 80-90% of time by skipping model training!

## 10.2 Skip Training AND Extraction - Use Saved Feature CSVs

**⚡⚡ ULTRA FAST MODE ⚡⚡**

If you've already extracted features and they're saved in CSVs, skip straight to fusion and LightGBM training!

In [ ]:
def run_from_saved_features(self) -> Dict:
    """Skip training AND extraction - use pre-extracted feature CSVs
    
    This is the FASTEST option when you've already extracted features and they're saved as CSVs.
    Goes straight to feature fusion and LightGBM training.
    
    Expected CSV locations:
        - CapsNet features: {output_dir}/features/capsnet/capsnet_features_fold_X_{day_folder}.csv
        - LSTM features: {output_dir}/features/lstm/lstm_features_fold_X_{day_folder}.csv
    
    Returns:
        Dictionary with complete pipeline results
    
    Example:
        pipeline = CompleteMLPipeline('7_24_data', 'pipeline_outputs', fast_mode=True)
        results = pipeline.run_from_saved_features()
        
    Time Savings:
        - Skip CapsNet training (~30-60 min per fold)
        - Skip LSTM training (~10-20 min per fold)
        - Skip feature extraction (~5-10 min per fold)
        - Total: ~90-95% faster! Only LightGBM training needed (~1-2 min per fold)
    """
    print(f"Starting Pipeline from Saved Feature CSVs for {self.day_folder}")
    print("=" * 60)
    print(" ⚡⚡ ULTRA FAST MODE - Skipping training AND extraction ⚡⚡")
    print(" Loading pre-extracted features from CSV files")
    print("=" * 60)
    
    try:
        import glob
        
        # Step 1: Find saved feature CSVs
        print(f"\nStep 1: Finding Saved Feature CSVs")
        
        capsnet_csv_pattern = f"{self.output_dir}/features/capsnet/capsnet_features_fold_*_{self.day_folder}.csv"
        lstm_csv_pattern = f"{self.output_dir}/features/lstm/lstm_features_fold_*_{self.day_folder}.csv"
        
        capsnet_csvs_found = sorted(glob.glob(capsnet_csv_pattern))
        lstm_csvs_found = sorted(glob.glob(lstm_csv_pattern))
        
        print(f"   Found {len(capsnet_csvs_found)} CapsNet feature CSVs")
        print(f"   Found {len(lstm_csvs_found)} LSTM feature CSVs")
        
        if len(capsnet_csvs_found) == 0 or len(lstm_csvs_found) == 0:
            raise FileNotFoundError(
                f"Could not find saved feature CSVs!\n"
                f"   CapsNet CSVs: {capsnet_csvs_found}\n"
                f"   LSTM CSVs: {lstm_csvs_found}\n"
                f"   Looking in: {self.output_dir}/features/\n"
                f"   Make sure you've extracted features first or check the paths.\n\n"
                f"   Expected paths:\n"
                f"      {capsnet_csv_pattern}\n"
                f"      {lstm_csv_pattern}"
            )
        
        # Map CSVs to folds
        capsnet_csvs = {}
        lstm_csvs = {}
        
        for i, (capsnet_path, lstm_path) in enumerate(zip(capsnet_csvs_found, lstm_csvs_found), 1):
            if i <= self.n_folds:
                capsnet_csvs[i] = capsnet_path
                lstm_csvs[i] = lstm_path
                print(f"   Fold {i}:")
                print(f"      CapsNet CSV: {capsnet_path}")
                print(f"      LSTM CSV: {lstm_path}")
        
        # Step 2: Load features and run fusion + LightGBM
        print(f"\nStep 2: Loading Features, Fusing, and Training LightGBM")
        print("=" * 60)
        
        fold_results = {}
        
        for fold in range(1, self.n_folds + 1):
            if fold not in capsnet_csvs or fold not in lstm_csvs:
                print(f"\n   Skipping Fold {fold} - CSVs not found")
                continue
                
            print(f"\n{'='*70}")
            print(f"FOLD {fold}/{self.n_folds} - FUSION & LIGHTGBM TRAINING")
            print(f"{'='*70}")
            
            # Load pre-extracted features from CSV
            print(f"\n   Step {fold}.1: Loading CapsNet Features from CSV")
            capsnet_df = pd.read_csv(capsnet_csvs[fold])
            print(f"      Loaded {len(capsnet_df)} samples with {len(capsnet_df.columns)} columns")
            
            print(f"\n   Step {fold}.2: Loading LSTM Features from CSV")
            lstm_df = pd.read_csv(lstm_csvs[fold])
            print(f"      Loaded {len(lstm_df)} samples with {len(lstm_df.columns)} columns")
            
            # Fuse features
            print(f"\n   Step {fold}.3: Fusing Features for Fold {fold}")
            fused_features = self.fuse_features_fold(capsnet_df, lstm_df, fold)
            
            # Train LightGBM
            print(f"\n   Step {fold}.4: Training LightGBM for Fold {fold}")
            fold_result = self.train_lightgbm_fold(fused_features, fold)
            fold_results[fold] = fold_result
            
            # Print fold summary
            print(f"\n   FOLD {fold} COMPLETE!")
            print(f"      RMSE: {fold_result['rmse']:.4f}")
            print(f"      MAE: {fold_result['mae']:.4f}")
            print(f"      R²: {fold_result['r2']:.4f}")
            print(f"      Adjusted R²: {fold_result['adj_r2']:.4f}")
            print(f"{'='*70}\n")
        
        # Store fold results
        self.fold_results = fold_results
        
        # Step 3: Analyze results
        print(f"\nStep 3: Analyzing Results Across All Folds")
        results = self.analyze_results()
        
        # Step 4: Create visualizations
        print(f"\nStep 4: Creating Visualizations")
        self.create_visualizations()
        
        print(f"\nPipeline from Saved Features Finished Successfully!")
        print("=" * 60)
        print(f"⚡⚡ MAXIMUM TIME SAVINGS - Skipped both training AND extraction! ⚡⚡")
        
        return results
        
    except Exception as e:
        print(f"\nPipeline failed: {e}")
        traceback.print_exc()
        return None

# Add the method to the class
CompleteMLPipeline.run_from_saved_features = run_from_saved_features
print("Ultra-fast skip method added! Use pipeline.run_from_saved_features() to skip training AND extraction.")

### Usage Example - Skip Training AND Extraction

**⚡⚡ Use this when you have pre-extracted feature CSVs! ⚡⚡**

This is the FASTEST option - only runs fusion and LightGBM training (~1-2 minutes per fold).

In [ ]:
# EXAMPLE: Skip training AND extraction - use saved feature CSVs
# ⚡⚡ ULTRA FAST MODE - Only runs fusion + LightGBM ⚡⚡

pipeline = CompleteMLPipeline(DAY_FOLDER, OUTPUT_DIR, fast_mode=FAST_MODE)
results = pipeline.run_from_saved_features()

# This will:
# 1. Load CapsNet features from CSV (84,770 patches with lat/lon)
# 2. Load LSTM features from CSV (7,694 locations)
# 3. Aggregate CapsNet patches by location (84,770 → 7,694)
# 4. Fuse features (64D CapsNet + 32D LSTM = 96D)
# 5. Train LightGBM (fast!)
# 6. Analyze and visualize results
# 
# ⚡⚡ Saves 90-95% of time! Perfect for:
#   - Experimenting with different fusion strategies
#   - Testing different LightGBM hyperparameters
#   - Rapid iteration without re-training models
#   - Quick result generation for presentations

### 🚀 Execution Mode Comparison

Choose the right mode for your needs:

| Mode | Function | What It Does | Time per Fold | Best For |
|------|----------|--------------|---------------|----------|
| **🐢 FULL** | `run_complete_pipeline()` | Train CapsNet + Train LSTM + Extract + Fuse + LightGBM | ~40-80 min | First run, final results |
| **⚡ SKIP TRAINING** | `run_from_saved_models()` | Extract + Fuse + LightGBM | ~10-15 min | Models exist, testing fusion |
| **⚡⚡ ULTRA FAST** | `run_from_saved_features()` | Fuse + LightGBM only | ~1-2 min | Features exist, rapid iteration |

**Typical Workflow:**
1. **First time:** Use `run_complete_pipeline()` to train everything
2. **After training:** Use `run_from_saved_models()` if you want to re-extract features with different settings
3. **Quick experiments:** Use `run_from_saved_features()` to test different:
   - Aggregation strategies (mean vs max vs median)
   - LightGBM hyperparameters
   - Fusion methods
   - Feature selections

**What Gets Saved:**
```
pipeline_outputs/
├── models/
│   ├── capsnet/*.pth          ← Used by run_from_saved_models()
│   └── lstm/*.pth             ← Used by run_from_saved_models()
├── features/
│   ├── capsnet/*.csv          ← Used by run_from_saved_features() ⚡⚡
│   ├── lstm/*.csv             ← Used by run_from_saved_features() ⚡⚡
│   └── fused/*.csv            ← Generated by all modes
└── models/lightgbm/*.txt      ← Generated by all modes
```

In [ ]:
# ========================================
# CHOOSE YOUR EXECUTION MODE
# ========================================

# Initialize pipeline (same for all modes)
pipeline = CompleteMLPipeline(DAY_FOLDER, OUTPUT_DIR, fast_mode=FAST_MODE)

# ----------------------------------------
# OPTION 1: Full Pipeline (slowest, most complete)
# ----------------------------------------
# Uncomment to run complete pipeline from scratch
# results = pipeline.run_complete_pipeline()

# ----------------------------------------
# OPTION 2: Skip Training (medium speed)
# ----------------------------------------
# Uncomment to load trained models and extract features
# results = pipeline.run_from_saved_models()

# ----------------------------------------
# OPTION 3: Skip Training AND Extraction (fastest!)
# ----------------------------------------
# Uncomment to load pre-extracted features and go straight to fusion
# results = pipeline.run_from_saved_features()

print("Choose one option above and uncomment it to run!")
print("\nCurrent configuration:")
print(f"   Day: {DAY_FOLDER}")
print(f"   Output: {OUTPUT_DIR}")
print(f"   Fast mode: {FAST_MODE}")
print(f"   Folds: {2 if FAST_MODE else 5}")